## Installing Libraries

In [1]:
#Installing Libraries
!pip install -qU datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 42.9 MB/s eta 0:00:00


In [2]:
#Installing Libraries
!pip install -qU accelerate

In [3]:
#Installing Libraries
!pip install -qU transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 84.6 MB/s eta 0:00:00


In [4]:
#Installing Libraries
!pip install -qU evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00


## Import the Libraries

In [22]:
from transformers import (
    Trainer,
    TrainingArguments,
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    DataCollatorWithPadding,
    pipeline) # Changed Pipeline to pipeline
from datasets import Dataset
import evaluate
import pandas as pd
import numpy as np
import os
os.environ['WANDB_DISABLED']='true'

## Import Dataset

In [23]:
train_df = pd.read_csv('train.csv.zip')
train_df.head()

,qtype,Question,Answer
0,susceptibility,Who is at risk for Lymphocytic Choriomeningiti...,LCMV infections can occur after exposure to fr...
1,symptoms,What are the symptoms of Lymphocytic Choriomen...,LCMV is most commonly recognized as causing ne...
2,susceptibility,Who is at risk for Lymphocytic Choriomeningiti...,Individuals of all ages who come into contact ...
3,exams and tests,How to diagnose Lymphocytic Choriomeningitis (...,"During the first phase of the disease, the mos..."
4,treatment,What are the treatments for Lymphocytic Chorio...,"Aseptic meningitis, encephalitis, or meningoen..."


In [24]:
print('number of medical answers: ', len(train_df))
print('number of patient questions: ', len(train_df))

number of medical answers:  16407
number of patient questions:  16407


## Create Dataset Object

In [25]:
#Obtain the unique medical qtype labels
unique_qtype = train_df['qtype'].drop_duplicates()
unique_qtype = list(unique_qtype)
print (unique_qtype)

['susceptibility', 'symptoms', 'exams and tests', 'treatment', 'prevention', 'information', 'frequency', 'complications', 'causes', 'research', 'outlook', 'considerations', 'inheritance', 'stages', 'genetic changes', 'support groups']


In [26]:
qtype = list(train_df['qtype'])
answer = list(train_df['Answer'])

#Replace the medical specialty labels with numerical labels
qtype = [unique_qtype.index(i) for i in qtype]
print(qtype)

[0, 1, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 2, 3, 6, 4, 5, 4, 5, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 4, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 0, 1, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 1, 0, 2, 5, 6, 1, 2, 3, 7, 4, 4, 0, 1, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 0, 0, 1, 3, 0, 1, 0, 2, 3, 4, 5, 6, 2, 3, 4, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 0, 1, 0, 2, 3, 4, 5, 0, 2, 3, 4, 3, 1, 4, 5, 1, 5, 4, 0, 1, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 8, 2, 1, 3, 1, 5, 5, 0, 2, 3, 4, 4, 5, 0, 2, 3, 4, 1, 5, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 1, 5, 0, 2, 3, 4, 2, 3, 5, 5, 4, 5, 0, 1, 0, 2, 3, 4, 4, 0, 1, 0, 2, 3, 4, 4, 0, 4, 4, 1, 5, 5, 1, 0, 4, 3, 2, 4, 3, 5, 9, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 5, 5, 0, 3, 4, 0, 1, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 1

In [27]:
qtype = list(train_df['qtype'])
question = list(train_df['Question'])

#Replace the medical specialty labels with numerical labels
qtype = [unique_qtype.index(i) for i in qtype]
print(qtype)

[0, 1, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 2, 3, 6, 4, 5, 4, 5, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 4, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 0, 1, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 1, 0, 2, 5, 6, 1, 2, 3, 7, 4, 4, 0, 1, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 0, 0, 1, 3, 0, 1, 0, 2, 3, 4, 5, 6, 2, 3, 4, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 0, 1, 0, 2, 3, 4, 5, 0, 2, 3, 4, 3, 1, 4, 5, 1, 5, 4, 0, 1, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 8, 2, 1, 3, 1, 5, 5, 0, 2, 3, 4, 4, 5, 0, 2, 3, 4, 1, 5, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 1, 5, 0, 2, 3, 4, 2, 3, 5, 5, 4, 5, 0, 1, 0, 2, 3, 4, 4, 0, 1, 0, 2, 3, 4, 4, 0, 4, 4, 1, 5, 5, 1, 0, 4, 3, 2, 4, 3, 5, 9, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 5, 5, 0, 3, 4, 0, 1, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 0, 2, 3, 4, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 10, 9, 5, 3, 1

In [48]:
answer_dataset = Dataset.from_dict(
    dict (
        text = answer,
        label = qtype
    )
)
print (answer_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 16407
})


In [49]:
question_dataset = Dataset.from_dict(
    dict (
        text = question,
        label = qtype
    )
)
print (question_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 16407
})


## Split the Dataset

In [50]:
answer_dataset = answer_dataset.train_test_split(test_size=0.2)
print (answer_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 13125
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3282
    })
})


In [51]:
question_dataset = question_dataset.train_test_split(test_size=0.2)
print (question_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 13125
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3282
    })
})


In [70]:
answer_dataset['train'][0]

{'text': 'SYNGAP1-related intellectual disability is a neurological disorder characterized by moderate to severe intellectual disability that is evident in early childhood. The earliest features are typically delayed development of speech and motor skills, such as sitting, standing, and walking. Many people with this condition have weak muscle tone (hypotonia), which contributes to the difficulty with motor skills. Some affected individuals lose skills they had already acquired (developmental regression). Other features of SYNGAP1-related intellectual disability include recurrent seizures (epilepsy), hyperactivity, and autism spectrum disorders, which are conditions characterized by impaired communication and social interaction; almost everyone with SYNGAP1-related intellectual disability develops epilepsy, and about half have an autism spectrum disorder.',
 'label': 5}

In [66]:
question_dataset['train'][0]

{'text': 'Is protein C deficiency inherited ?', 'label': 12}

## Loading the Tokenizer

In [71]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [78]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

In [79]:
tokenized_dataset_answer = answer_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/13125 [00:00<?, ? examples/s]

Map:   0%|          | 0/3282 [00:00<?, ? examples/s]

In [80]:
tokenized_dataset_question = question_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/13125 [00:00<?, ? examples/s]

Map:   0%|          | 0/3282 [00:00<?, ? examples/s]

In [81]:
tokenized_dataset_answer['train'][0]

{'text': 'SYNGAP1-related intellectual disability is a neurological disorder characterized by moderate to severe intellectual disability that is evident in early childhood. The earliest features are typically delayed development of speech and motor skills, such as sitting, standing, and walking. Many people with this condition have weak muscle tone (hypotonia), which contributes to the difficulty with motor skills. Some affected individuals lose skills they had already acquired (developmental regression). Other features of SYNGAP1-related intellectual disability include recurrent seizures (epilepsy), hyperactivity, and autism spectrum disorders, which are conditions characterized by impaired communication and social interaction; almost everyone with SYNGAP1-related intellectual disability develops epilepsy, and about half have an autism spectrum disorder.',
 'label': 5,
 'input_ids': [101,
  19962,
  3654,
  2361,
  2487,
  1011,
  3141,
  7789,
  11980,
  2003,
  1037,
  23130,
  8761

In [82]:
tokenized_dataset_question['train'][0]

{'text': 'Is protein C deficiency inherited ?',
 'label': 12,
 'input_ids': [101, 2003, 5250, 1039, 18888, 7900, 1029, 102],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}

## Creating the Data Collator

In [83]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Loading the Pre-trained Model

In [84]:
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=len(unique_qtype))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Fine-tuning the Pre-trained Model

In [85]:
#Map model's output predictions (integer indices) back t0 their corresponding label names.
model.config.id2label = {i: label for i, label in enumerate(unique_qtype)}
print(model.config.id2label)

{0: 'susceptibility', 1: 'symptoms', 2: 'exams and tests', 3: 'treatment', 4: 'prevention', 5: 'information', 6: 'frequency', 7: 'complications', 8: 'causes', 9: 'research', 10: 'outlook', 11: 'considerations', 12: 'inheritance', 13: 'stages', 14: 'genetic changes', 15: 'support groups'}


In [86]:
from torch import logit
metric = evaluate.load('accuracy')

In [87]:
epochs = 8
#Define the training arguments
training_args = TrainingArguments(
    output_dir = './results',
    num_train_epochs=epochs,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    load_best_model_at_end=True,

    warmup_steps = len(tokenized_dataset_answer['train']) // 5,
    weight_decay = 0.05,

    logging_steps =1,
    log_level = 'info',
    eval_strategy= 'epoch',
    save_strategy= 'epoch',
)


In [88]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

#Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset_answer['train'],
    eval_dataset=tokenized_dataset_answer['test'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [89]:
trainer.train()

The following columns in the Training set don't have a corresponding argument in `DistilBertForSequenceClassification.forward` and have been ignored: text. If text are not expected by `DistilBertForSequenceClassification.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 13,125
  Num Epochs = 8
  Num update steps per epoch = 411
  Instantaneous batch size per device = 32
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 1
  Total optimization steps = 3,288
  Number of trainable parameters = 66,965,776


Epoch,Training Loss,Validation Loss,Accuracy
1,1.028293,0.659093,0.872943
2,0.053170,0.227915,0.944546
3,0.665099,0.167919,0.963437
4,0.004697,0.159314,0.961304
5,0.000735,0.177575,0.960695
6,0.001438,0.190670,0.966484
7,0.027644,0.190331,0.968921
8,0.001151,0.190296,0.968007


The following columns in the Evaluation set don't have a corresponding argument in `DistilBertForSequenceClassification.forward` and have been ignored: text. If text are not expected by `DistilBertForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 3282
  Batch size = 32
Saving model checkpoint to ./results/checkpoint-411
Configuration saved in ./results/checkpoint-411/config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in ./results/checkpoint-411/model.safetensors
Saving Trainer.data_collator.tokenizer by default as Trainer.processing_class is `None`
tokenizer config file saved in ./results/checkpoint-411/tokenizer_config.json
The following columns in the Evaluation set don't have a corresponding argument in `DistilBertForSequenceClassification.forward` and have been ignored: text. If text are not expected by `DistilBertForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 3282
  Batch size = 32
Saving model checkpoint to ./results/checkpoint-822
Configuration saved in ./results/checkpoint-822/config.json


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in ./results/checkpoint-2877/model.safetensors
Saving Trainer.data_collator.tokenizer by default as Trainer.processing_class is `None`
tokenizer config file saved in ./results/checkpoint-2877/tokenizer_config.json
The following columns in the Evaluation set don't have a corresponding argument in `DistilBertForSequenceClassification.forward` and have been ignored: text. If text are not expected by `DistilBertForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 3282
  Batch size = 32
Saving model checkpoint to ./results/checkpoint-3288
Configuration saved in ./results/checkpoint-3288/config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in ./results/checkpoint-3288/model.safetensors
Saving Trainer.data_collator.tokenizer by default as Trainer.processing_class is `None`
tokenizer config file saved in ./results/checkpoint-3288/tokenizer_config.json


Training completed. Do not forget to share your model on huggingface.co/models =)


Loading best model from ./results/checkpoint-1644 (score: 0.15931396186351776).
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=3288, training_loss=0.32125648364671205, metrics={'train_runtime': 5650.0951, 'train_samples_per_second': 18.584, 'train_steps_per_second': 0.582, 'total_flos': 1.38762024865872e+16, 'train_loss': 0.32125648364671205, 'epoch': 8.0})

## Prediction and Evaluation

In [90]:
pred_output = trainer.predict(tokenized_dataset_answer['test'])
print(pred_output.metrics)

The following columns in the test set don't have a corresponding argument in `DistilBertForSequenceClassification.forward` and have been ignored: text. If text are not expected by `DistilBertForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 3282
  Batch size = 32


{'test_loss': 0.1591932624578476, 'test_accuracy': 0.9606946983546618, 'test_runtime': 56.4796, 'test_samples_per_second': 58.109, 'test_steps_per_second': 1.824}


In [91]:
pred_output = trainer.predict(tokenized_dataset_question['test'])
print(pred_output.metrics)

The following columns in the test set don't have a corresponding argument in `DistilBertForSequenceClassification.forward` and have been ignored: text. If text are not expected by `DistilBertForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 3282
  Batch size = 32


{'test_loss': 1.3816362619400024, 'test_accuracy': 0.6337599024984766, 'test_runtime': 3.0058, 'test_samples_per_second': 1091.9, 'test_steps_per_second': 34.267}


In [92]:
answer_dataset['test'][2]

{'text': "GRACILE syndrome is a severe disorder that begins before birth. GRACILE stands for the condition's characteristic features: growth retardation, aminoaciduria, cholestasis, iron overload, lactic acidosis, and early death.  In GRACILE syndrome, growth before birth is slow (intrauterine growth retardation). Affected newborns are smaller than average and have an inability to grow and gain weight at the expected rate (failure to thrive). A characteristic of GRACILE syndrome is excess iron in the liver, which likely begins before birth. Iron levels may begin to improve after birth, although they typically remain elevated. Within the first day of life, infants with GRACILE syndrome have a buildup of a chemical called lactic acid in the body (lactic acidosis). They also have kidney problems that lead to an excess of molecules called amino acids in the urine (aminoaciduria). Babies with GRACILE syndrome have cholestasis, which is a reduced ability to produce and release a digestive fl

In [93]:
question_dataset['test'][2]

{'text': 'How many people are affected by septo-optic dysplasia ?', 'label': 6}

In [96]:
pipe = pipeline('text-classification', model=model, tokenizer=tokenizer)
pipe("""GRACILE syndrome is a severe disorder that begins before birth. GRACILE stands for the condition's characteristic features: growth retardation, aminoaciduria, cholestasis, iron overload, lactic acidosis, and early death.  In GRACILE syndrome, growth before birth is slow (intrauterine growth retardation). Affected newborns are smaller than average and have an inability to grow and gain weight at the expected rate (failure to thrive). A characteristic of GRACILE syndrome is excess iron in the liver, which likely begins before birth. Iron levels may begin to improve after birth, although they typically remain elevated. Within the first day of life, infants with GRACILE syndrome have a buildup of a chemical called lactic acid in the body (lactic acidosis). They also have kidney problems that lead to an excess of molecules called amino acids in the urine (aminoaciduria). Babies with GRACILE syndrome have cholestasis, which is a reduced ability to produce and release a digestive fluid called bile. Cholestasis leads to irreversible liver disease (cirrhosis) in the first few months of life.  Because of the severe health problems caused by GRACILE syndrome, infants with this condition do not survive for more than a few months, and about half die within a few days of birth.""")

[{'label': 'information', 'score': 0.9988634586334229}]

In [97]:
pipe = pipeline('text-classification', model=model, tokenizer=tokenizer)
pipe('How many people are affected by septo-optic dysplasia ?')

[{'label': 'causes', 'score': 0.45962417125701904}]

## Saving the Fine-tuned Model

In [98]:
trainer.save_model()

Saving model checkpoint to ./results
Configuration saved in ./results/config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in ./results/model.safetensors
Saving Trainer.data_collator.tokenizer by default as Trainer.processing_class is `None`
tokenizer config file saved in ./results/tokenizer_config.json


In [99]:
pipe = pipeline('text-classification','./results', tokenizer=tokenizer)
pipe("""GRACILE syndrome is a severe disorder that begins before birth. GRACILE stands for the condition's characteristic features: growth retardation, aminoaciduria, cholestasis, iron overload, lactic acidosis, and early death.  In GRACILE syndrome, growth before birth is slow (intrauterine growth retardation). Affected newborns are smaller than average and have an inability to grow and gain weight at the expected rate (failure to thrive). A characteristic of GRACILE syndrome is excess iron in the liver, which likely begins before birth. Iron levels may begin to improve after birth, although they typically remain elevated. Within the first day of life, infants with GRACILE syndrome have a buildup of a chemical called lactic acid in the body (lactic acidosis). They also have kidney problems that lead to an excess of molecules called amino acids in the urine (aminoaciduria). Babies with GRACILE syndrome have cholestasis, which is a reduced ability to produce and release a digestive fluid called bile. Cholestasis leads to irreversible liver disease (cirrhosis) in the first few months of life.  Because of the severe health problems caused by GRACILE syndrome, infants with this condition do not survive for more than a few months, and about half die within a few days of birth.""")

loading configuration file ./results/config.json
Model config DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForSequenceClassification"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_dim": 3072,
  "id2label": {
    "0": "susceptibility",
    "1": "symptoms",
    "2": "exams and tests",
    "3": "treatment",
    "4": "prevention",
    "5": "information",
    "6": "frequency",
    "7": "complications",
    "8": "causes",
    "9": "research",
    "10": "outlook",
    "11": "considerations",
    "12": "inheritance",
    "13": "stages",
    "14": "genetic changes",
    "15": "support groups"
  },
  "initializer_range": 0.02,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_10": 10,
    "LABEL_11": 11,
    "LABEL_12": 12,
    "LABEL_13": 13,
    "LABEL_14": 14,
    "LABEL_15": 15,
    "LABEL_2": 2,
    "LABEL_3": 3,
    "LABEL_4": 4,
    "LABEL_5": 5

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'information', 'score': 0.9988634586334229}]

In [100]:
pipe = pipeline('text-classification','./results', tokenizer=tokenizer)
pipe('How many people are affected by septo-optic dysplasia ?')

loading configuration file ./results/config.json
Model config DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForSequenceClassification"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_dim": 3072,
  "id2label": {
    "0": "susceptibility",
    "1": "symptoms",
    "2": "exams and tests",
    "3": "treatment",
    "4": "prevention",
    "5": "information",
    "6": "frequency",
    "7": "complications",
    "8": "causes",
    "9": "research",
    "10": "outlook",
    "11": "considerations",
    "12": "inheritance",
    "13": "stages",
    "14": "genetic changes",
    "15": "support groups"
  },
  "initializer_range": 0.02,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_10": 10,
    "LABEL_11": 11,
    "LABEL_12": 12,
    "LABEL_13": 13,
    "LABEL_14": 14,
    "LABEL_15": 15,
    "LABEL_2": 2,
    "LABEL_3": 3,
    "LABEL_4": 4,
    "LABEL_5": 5

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'causes', 'score': 0.45962417125701904}]